In [24]:
%pip install osmnx folium

   ---------------------------------------- 0.0/113.4 kB ? eta -:--:--
   ---------- ----------------------------- 30.7/113.4 kB 1.3 MB/s eta 0:00:01
   -------------------------------- ------- 92.2/113.4 kB 1.3 MB/s eta 0:00:01
   -------------------------------------- 113.4/113.4 kB 937.8 kB/s eta 0:00:00
   ---------------------------------------- 0.0/93.9 kB ? eta -:--:--
   ---------------------------------------- 93.9/93.9 kB 2.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [31]:
import folium
import pandas as pd
import ast

print("Đang khởi tạo bản đồ tương tác Folium...")

nodes_df = pd.read_csv('data/nodes.csv')
edges_df = pd.read_csv('data/edges.csv')
# 1. Đọc lại dữ liệu và index (Nếu bạn đã load ở trên thì có thể bỏ qua bước đọc file)
nodes_df = nodes_df.set_index('osmid')
edges_df = edges_df.copy()

# Khôi phục mảng list (nếu bạn đã chạy ép kiểu ở trên rồi thì comment dòng này lại)
if isinstance(edges_df['los'].iloc[0], str):
    edges_df['los'] = edges_df['los'].apply(ast.literal_eval)

# 2. Lấy trọng số LOS ở khung giờ 17:30 (17h * 2 + 1 = 35)
peak_idx = 2
edges_df['current_los'] = edges_df['los'].apply(lambda x: x[peak_idx])

# 3. Khởi tạo bản đồ Folium ở tọa độ trung tâm TP.HCM
center_lat = nodes_df['y'].mean()
center_lon = nodes_df['x'].mean()
# Dùng tiles='CartoDB dark_matter' để tạo nền đen cho bản đồ
m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles='CartoDB dark_matter')

# 4. Hàm quy định thang màu tắc đường (Tăng dần)
def get_traffic_style(los):
    if los <= 1.3:
        return '#00FF00', 1.0  # Xanh lá (Thông thoáng) - Nét mảnh
    elif los <= 1.5:
        return '#FFFF00', 2.0  # Vàng (Hơi chậm) - Nét vừa
    elif los <= 1.7:
        return '#FFA500', 3.0  # Cam (Ùn ứ) - Nét to
    else:
        return '#FF0000', 4.0  # Đỏ (Kẹt cứng) - Nét rất to

# 5. Vẽ từng đoạn đường lên bản đồ
for _, row in edges_df.iterrows():
    try:
        # Tọa độ Vĩ độ (y) - Kinh độ (x) của điểm u và v
        u_coord = (nodes_df.loc[row['u'], 'y'], nodes_df.loc[row['u'], 'x'])
        v_coord = (nodes_df.loc[row['v'], 'y'], nodes_df.loc[row['v'], 'x'])
        
        # Lấy màu và độ dày nét vẽ dựa trên độ kẹt xe
        color, weight = get_traffic_style(row['current_los'])
        
        # Vẽ đoạn thẳng nối 2 nút giao
        folium.PolyLine(
            locations=[u_coord, v_coord],
            color=color,
            weight=weight,
            opacity=0.8
        ).add_to(m)
        
    except KeyError:
        continue # Bỏ qua nếu nhiễu node

file_name = "bandoketxe_17h30.html"
m.save(file_name)

Đang khởi tạo bản đồ tương tác Folium...
